# 🛡️ BƯỚC 3 — Baseline 1 (B1): Phòng Thủ Nhiễu Gaussian tại Cut Layer

### 🎯 Mục tiêu & Ý nghĩa Khoa học:
- **Khung đánh giá thống nhất (Unified Protocol cho B1–B6)**:
  1. **Pha 1**: Huấn luyện Split Learning có phòng thủ nhiễu Gauss $z' = z + \sigma \cdot \epsilon$ với $\sigma \in \{0.1, 0.5, 1.0\}$ (100 epochs).
  2. **Pha 2**: Đóng băng Client & phòng thủ, huấn luyện Decoder tái tạo thích ứng (**Adaptive Reconstruction Attacker**) trên IR có nhiễu (30 epochs).
  3. **Pha 3**: Đo lường sự đánh đổi giữa **Utility (Test Accuracy)** và **Security (PSNR, SSIM, LPIPS)**.
- **Điểm mấu chốt đối chứng cho Luận văn**:
  - Nhiễu Gaussian làm giảm chất lượng ảnh (PSNR/SSIM giảm mạnh) nhưng **không chống được đối thủ thích ứng** — Decoder vẫn tái tạo được cấu trúc và hình dáng ngữ nghĩa.
  - Đây là bằng chứng thực nghiệm quan trọng chứng minh cơ chế phòng thủ đơn giản thất bại, khẳng định tính tất yếu của **Task-Aware Perceptual Encryption (Bước 5)**.

---  
## 1. Kiểm tra Môi trường & GPU (Tesla T4 / V100 / A100)

In [ ]:
!nvidia-smi

import torch
print(f"\nPyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Thiết bị GPU    : {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ CẢNH BÁO: Hãy bật GPU trong 'Runtime' -> 'Change runtime type' -> 'T4 GPU'!")

---  
## 2. Kết nối Google Drive (Lưu Checkpoints & Bảng Kết quả vĩnh viễn)

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
DRIVE_STEP3_DIR = '/content/drive/MyDrive/AbReTAPE_Step3'
os.makedirs(DRIVE_STEP3_DIR, exist_ok=True)
print(f"✅ Thư mục lưu kết quả Bước 3 trên Google Drive: {DRIVE_STEP3_DIR}")

---  
## 3. Cài đặt Thư viện Phụ trợ & Kiểm tra Mã Nguồn

In [ ]:
!pip install -q -r requirements.txt

import os
if not os.path.exists('src'):
    !git clone https://github.com/CuongBien/AbReTAPE.git /content/AbReTAPE
    %cd /content/AbReTAPE
else:
    print("✅ Thư mục mã nguồn 'src' đã sẵn sàng!")

!ls -la src


---  
## 4. Tải Dữ liệu CIFAR-10 & Chạy Kiểm thử Đơn vị (Smoke Test Bước 3)

In [ ]:
# Chạy kiểm thử đơn vị cho toàn bộ pipeline Bước 3
!python run_tests.py


---  
## 5. Huấn luyện Đơn lẻ 1 Mức Nhiễu (Ví dụ: $\sigma = 0.5$)
> **Gợi ý**: Chạy cell này nếu bạn muốn chạy thử nghiệm nhanh một mức nhiễu cụ thể.

In [ ]:
!python run_step3_baselines.py --defense b1 \
    --sigma 0.5 \
    --epochs 100 \
    --decoder-epochs 30 \
    --batch-size 128 \
    --lr 0.1 \
    --decoder-lr 0.001 \
    --eval-freq 5 \
    --data-dir data \
    --output-dir /content/drive/MyDrive/AbReTAPE_Step3


---  
## 6. Chạy Quét Toàn Diện Đa Mức Nhiễu (Sweep $\sigma \in \{0.1, 0.5, 1.0\}$)
- Tự động huấn luyện Split Learning (100 epoch) và Decoder tấn công (30 epoch) cho cả 3 mức nhiễu.
- Xuất bảng so sánh tổng hợp `results_b1.csv`, đồ thị `b1_tradeoff_curves.png` và lưới ảnh `b1_reconstruction_comparison.png`.

In [ ]:
!python run_step3_baselines.py --defense b1 \
    --sweep \
    --sigmas 0.1,0.5,1.0 \
    --epochs 100 \
    --decoder-epochs 30 \
    --batch-size 128 \
    --lr 0.1 \
    --decoder-lr 0.001 \
    --eval-freq 5 \
    --data-dir data \
    --output-dir /content/drive/MyDrive/AbReTAPE_Step3


---  
## 7. Trực quan hóa Đồ thị Privacy-Utility Trade-off across $\sigma$

In [ ]:
from IPython.display import Image, display
import os

tradeoff_file = "/content/drive/MyDrive/AbReTAPE_Step3/b1_tradeoff_curves.png"
if not os.path.exists(tradeoff_file):
    tradeoff_file = "output/AbReTAPE_Step3/b1_tradeoff_curves.png"

if os.path.exists(tradeoff_file):
    print("📈 ĐỒ THỊ PRIVACY - UTILITY TRADE-OFF (ACCURACY vs PSNR / SSIM / LPIPS):")
    display(Image(filename=tradeoff_file, width=950))
else:
    print(f"⚠️ Chưa tìm thấy file đồ thị tại: {tradeoff_file}")

---  
## 8. Trực quan hóa Lưới Ảnh So Sánh Tái Tạo Trực Quan

In [ ]:
recons_file = "/content/drive/MyDrive/AbReTAPE_Step3/b1_reconstruction_comparison.png"
if not os.path.exists(recons_file):
    recons_file = "output/AbReTAPE_Step3/b1_tradeoff_curves.png"

if os.path.exists(recons_file):
    print("🖼️ LƯỚI ẢNH SO SÁNH CHẤT LƯỢNG TÁI TẠO (GỐC vs CÁC MỨC NHIỄU):")
    display(Image(filename=recons_file, width=950))
else:
    print(f"⚠️ Chưa tìm thấy file lưới ảnh tại: {recons_file}")

---  
## 9. Hiển thị Bảng Tổng Hợp Kết Quả B1 (Pandas DataFrame)

In [ ]:
import pandas as pd
import json
import os

csv_path = "/content/drive/MyDrive/AbReTAPE_Step3/results_b1.csv"
if not os.path.exists(csv_path):
    csv_path = "output/AbReTAPE_Step3/results_b1.csv"
if not os.path.exists(csv_path):
    csv_path = "output/AbReTAPE_Step3/results_b1.json"

if os.path.exists(csv_path):
    if csv_path.endswith('.json'):
        with open(csv_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        df = pd.DataFrame(data)
    else:
        df = pd.read_csv(csv_path)
    df['test_acc'] = (df['test_acc'] * 100).round(2).astype(str) + '%'
    df['psnr'] = df['psnr'].round(2).astype(str) + ' dB'
    df['ssim'] = df['ssim'].round(4)
    if 'lpips' in df.columns and df['lpips'].notnull().any():
        df['lpips'] = df['lpips'].round(4)
    print("📊 BẢNG TỔNG HỢP KẾT QUẢ BASELINE B1 (GAUSSIAN NOISE):")
    display(df)
else:
    print(f"⚠️ Chưa tìm thấy file kết quả tại: {csv_path}")
